In [27]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm import tqdm
import torchvision.transforms as transforms

# AMP Imports for RTX 4060 optimization
from torch.amp import autocast, GradScaler

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import torchvision.transforms as transforms

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Compute Device: {device.upper()}")

# 1. Load from Cache (No Redownloading!)
print("⏳ Loading Synthetic Dataset from Local Cache...")
# Using streaming=True prevents RAM overflow
train_dataset = load_dataset("princeton-vl/LayeredDepth-Syn", split="train", streaming=True)

# 2. Strict Key Verification based on your findings
sample = next(iter(train_dataset))
keys = list(sample.keys())
print(f"✅ Dataset Keys Verified: {keys}")

if 'image.png' not in keys or 'depth_1.png' not in keys or 'depth_2.png' not in keys:
    print("⚠️ WARNING: The expected keys were not found in this sample!")
else:
    print("✅ Ready to train. No 'depth_0.png' found, mapping layers correctly.")

✅ Compute Device: CUDA
⏳ Loading Synthetic Dataset from Local Cache...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

✅ Dataset Keys Verified: ['image.png', 'depth_1.png', 'depth_2.png', 'depth_3.png', 'depth_4.png', 'depth_5.png', 'depth_6.png', 'depth_7.png', 'depth_8.png', '__key__']
✅ Ready to train. No 'depth_0.png' found, mapping layers correctly.


In [2]:
# Transforms
transform_img = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_depth = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Scale-Invariant Logarithmic Loss (As per Paper 1)
class SILogLoss(nn.Module):
    def __init__(self, lambd=0.5):
        super().__init__()
        self.lambd = lambd

    def forward(self, pred, target):
        if pred.shape != target.shape:
            pred = F.interpolate(pred, size=target.shape[-2:], mode='bilinear', align_corners=False)
            
        valid_mask = (target > 0).detach()
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True).to(pred.device)

        diff_log = torch.log(pred[valid_mask] + 1e-6) - torch.log(target[valid_mask] + 1e-6)
        loss = torch.sqrt(torch.pow(diff_log, 2).mean() - self.lambd * torch.pow(diff_log.mean(), 2))
        return loss

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [4]:
class ChannelAttention(nn.Module):
    """
    Channel Attention (SE-like module)
    Learns which feature channels are most important
    
    Args:
        num_feat: Number of feature channels
        squeeze_factor: Channel reduction ratio
    """
    def __init__(self, num_feat, squeeze_factor=16):
        super(ChannelAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Conv2d(num_feat, num_feat // squeeze_factor, 1, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_feat // squeeze_factor, num_feat, 1, padding=0),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        Args:
            x: Input features [B, C, H, W]
        
        Returns:
            output: Channel-weighted features [B, C, H, W]
        """
        y = self.attention(x)
        return x * y


class CAB(nn.Module):
    """
    Channel Attention Block
    Combines convolution with channel attention
    
    Args:
        num_feat: Number of features
        compress_ratio: Compression ratio for convolutions
        squeeze_factor: Channel squeeze factor for attention
    """
    def __init__(self, num_feat, compress_ratio=3, squeeze_factor=30):
        super(CAB, self).__init__()
        self.cab = nn.Sequential(
            nn.Conv2d(num_feat, num_feat // compress_ratio, 3, 1, 1),
            nn.GELU(),
            nn.Conv2d(num_feat // compress_ratio, num_feat, 3, 1, 1),
            ChannelAttention(num_feat, squeeze_factor)
        )

    def forward(self, x):
        return self.cab(x)


class Mlp(nn.Module):
    """
    MLP (Feed-Forward Network)
    Used after attention layers
    
    Args:
        in_features: Input feature dimension
        hidden_features: Hidden layer dimension
        out_features: Output feature dimension
        act_layer: Activation function
        drop: Dropout rate
    """
    def __init__(self, in_features, hidden_features=None, out_features=None, 
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def window_partition(x, window_size):
    """
    Partition feature map into non-overlapping windows
    
    Args:
        x: Input features [B, H, W, C]
        window_size: Window size
    
    Returns:
        windows: Windowed features [num_windows*B, window_size, window_size, C]
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Reverse window partition
    
    Args:
        windows: Windowed features [num_windows*B, window_size, window_size, C]
        window_size: Window size
        H: Height of feature map
        W: Width of feature map
    
    Returns:
        x: Original feature map [B, H, W, C]
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    """
    Window-based Multi-head Self Attention
    Processes features within local windows
    
    Args:
        dim: Number of input channels
        window_size: Window size (tuple or int)
        num_heads: Number of attention heads
        qkv_bias: If True, add learnable bias to q, k, v
    """
    def __init__(self, dim, window_size, num_heads, qkv_bias=True):
        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # QKV projection
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        """
        Args:
            x: Input features [num_windows*B, N, C]
        
        Returns:
            output: Attention-processed features [num_windows*B, N, C]
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [B_, num_heads, N, head_dim]

        # Scaled dot-product attention
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))  # [B_, num_heads, N, N]
        attn = F.softmax(attn, dim=-1)

        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x


In [5]:
class HAB(nn.Module):
    """
    Hybrid Attention Block
    Combines window attention with channel attention
    
    Args:
        dim: Number of input channels
        num_heads: Number of attention heads
        window_size: Window size for local attention
        mlp_ratio: Ratio of MLP hidden dim to embedding dim
    """
    def __init__(self, dim, num_heads, window_size=8, mlp_ratio=4.):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.mlp_ratio = mlp_ratio

        # Layer normalization
        self.norm1 = nn.LayerNorm(dim)
        
        # Window-based self-attention
        self.attn = WindowAttention(
            dim, 
            window_size=(window_size, window_size), 
            num_heads=num_heads
        )

        # Channel Attention Block (CAB)
        self.conv_block = CAB(num_feat=dim)

        # MLP
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim)

    def forward(self, x):
        """
        Args:
            x: Input features [B, C, H, W]
        
        Returns:
            output: Processed features [B, C, H, W]
        """
        B, C, H, W = x.shape
        
        shortcut = x
        
        # Reshape for attention: [B, C, H, W] -> [B, H, W, C]
        x = x.permute(0, 2, 3, 1).contiguous()
        
        # Apply LayerNorm
        x = self.norm1(x)
        
        # Partition into windows
        x_windows = window_partition(x, self.window_size)  # [nW*B, window_size, window_size, C]
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # [nW*B, window_size*window_size, C]
        
        # Window attention
        attn_windows = self.attn(x_windows)  # [nW*B, window_size*window_size, C]
        
        # Merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_windows, self.window_size, H, W)  # [B, H, W, C]
        
        # Reshape back: [B, H, W, C] -> [B, C, H, W]
        x = x.permute(0, 3, 1, 2).contiguous()
        
        # Residual connection
        x = shortcut + x
        
        # Channel attention
        conv_x = self.conv_block(x)
        x = x + conv_x
        
        # MLP with residual
        # Reshape for MLP
        shortcut = x
        x = x.permute(0, 2, 3, 1).contiguous()  # [B, H, W, C]
        x = x.view(B, H * W, C)  # [B, H*W, C]
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()  # [B, C, H, W]
        x = shortcut + x
        
        return x


In [6]:
class RHAG(nn.Module):
    """
    Residual Hybrid Attention Group
    Stack of HAB blocks with residual connection
    
    Args:
        dim: Number of input channels
        num_heads: Number of attention heads
        num_blocks: Number of HAB blocks
        window_size: Window size for local attention
        mlp_ratio: MLP expansion ratio
    """
    def __init__(self, dim, num_heads=8, num_blocks=3, window_size=8, mlp_ratio=4.):
        super().__init__()
        
        # Stack of Hybrid Attention Blocks
        self.blocks = nn.ModuleList([
            HAB(dim, num_heads, window_size, mlp_ratio)
            for _ in range(num_blocks)
        ])
        
        # Residual connection with 1x1 conv
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)

    def forward(self, x):
        """
        Args:
            x: Input features [B, C, H, W]
        
        Returns:
            output: Processed features [B, C, H, W]
        """
        shortcut = x
        
        # Process through HAB blocks
        for block in self.blocks:
            x = block(x)
        
        # Apply convolution and add residual
        x = self.conv(x) + shortcut
        
        return x


In [ ]:
# ==========================================
# 2. SFIN BLOCK (Spatial-Frequency Interactive Network)
# ==========================================
class FourierUnit(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_layer = nn.Conv2d(in_channels * 2 + 2, out_channels * 2, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels * 2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        orig_dtype = x.dtype
        # Force to FP32 to bypass the cuFFT Power-of-Two hardware requirement
        x_float = x.to(torch.float32)
        
        with autocast('cuda', enabled=False):
            batch = x_float.shape[0]
            ffted = torch.fft.rfftn(x_float, dim=(-2, -1), norm='ortho')
            ffted = torch.stack((ffted.real, ffted.imag), dim=-1).permute(0, 1, 4, 2, 3).contiguous()
            ffted = ffted.view((batch, -1,) + ffted.size()[3:])
            
            h, w = ffted.shape[-2:]
            cv = torch.linspace(0, 1, h, device=x.device)[None, None, :, None].expand(batch, 1, h, w)
            ch = torch.linspace(0, 1, w, device=x.device)[None, None, None, :].expand(batch, 1, h, w)
            
            ffted = self.relu(self.bn(self.conv_layer(torch.cat((cv, ch, ffted), dim=1))))
            ffted = ffted.view((batch, -1, 2,) + ffted.size()[2:]).permute(0, 1, 3, 4, 2).contiguous()
            ffted = torch.complex(ffted[..., 0], ffted[..., 1])
            
            output = torch.fft.irfftn(ffted, s=x.shape[-2:], dim=(-2, -1), norm='ortho')
            
        return output.to(orig_dtype)

class SpectralTransform(nn.Module):
    def __init__(self, channels):
        super(SpectralTransform, self).__init__()
        self.conv1 = nn.Conv2d(channels // 2, channels // 2, 3, 1, 1)
        self.fu = FourierUnit(channels // 2, channels // 2)
        self.conv2 = nn.Conv2d(channels, channels // 2, 3, 1, 1)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.fu(x1)
        x = self.conv2(torch.cat([x, x2], dim=1))
        return x

class FFC(nn.Module):
    def __init__(self, channels):
        super(FFC, self).__init__()
        self.convl2l = nn.Conv2d(channels // 2, channels // 2, 3, 1, 1)
        self.convl2g = nn.Conv2d(channels // 2, channels // 2, 3, 1, 1)
        self.convg2l = nn.Conv2d(channels // 2, channels // 2, 3, 1, 1)
        self.convg2g = SpectralTransform(channels)

    def forward(self, x):
        if type(x) is tuple:
            x_l, x_g = x
        else:
            x_l, x_g = x, 0
        out_xl = self.convl2l(x_l) + self.convg2l(x_g)
        out_xg = self.convl2g(x_l) + self.convg2g(x_g)
        return out_xl, out_xg

class SFIB(nn.Module):
    def __init__(self, channels):
        super(SFIB, self).__init__()
        self.ffc = FFC(channels)
        self.bn_l = nn.BatchNorm2d(channels // 2)
        self.bn_g = nn.BatchNorm2d(channels // 2)
        self.act_l = nn.ReLU(inplace=True)
        self.act_g = nn.ReLU(inplace=True)

    def forward(self, x):
        x_l, x_g = self.ffc(x)
        x_l = self.act_l(self.bn_l(x_l))
        x_g = self.act_g(self.bn_g(x_g))
        return x_l, x_g

class SFINResBlock(nn.Module):
    def __init__(self, channels):
        super(SFINResBlock, self).__init__()
        self.conv1 = SFIB(channels)
        self.conv2 = SFIB(channels)

    def forward(self, x):
        x_l, x_g = torch.split(x, (x.shape[1] // 2, x.shape[1] // 2), dim=1)
        id_l, id_g = x_l, x_g
        x_l, x_g = self.conv1((x_l, x_g))
        x_l, x_g = self.conv2((x_l, x_g))
        x_l, x_g = id_l + x_l, id_g + x_g
        out = torch.cat((x_l, x_g), dim=1)
        return out

class SFINBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks=2):
        super(SFINBlock, self).__init__()
        self.input_conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        blocks = []
        for _ in range(num_blocks):
            blocks.append(SFINResBlock(out_channels))
        self.blocks = nn.Sequential(*blocks)
        self.output_conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)

    def forward(self, x):
        x = self.input_conv(x)
        shortcut = x
        x = self.blocks(x)
        x = x + shortcut  
        x = self.output_conv(x)
        return x

In [29]:
# ==========================================
# 3. ATTENTION GATE 
# ==========================================
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g_shape = g.shape
        x_shape = x.shape
        if g_shape[2:] != x_shape[2:]:
            g = F.interpolate(g, size=x_shape[2:], mode='bilinear', align_corners=False)
        g1 = self.W_g(g)  
        x1 = self.W_x(x)  
        psi = self.relu(g1 + x1)  
        psi = self.psi(psi)  
        output = x * psi  
        return output

class AttentionGateSimple(nn.Module):
    def __init__(self, F_g, F_l):
        super(AttentionGateSimple, self).__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(F_g + F_l, F_l // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(F_l // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(F_l // 2, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=False)
        combined = torch.cat([g, x], dim=1)  
        attention_map = self.attention(combined)  
        output = x * attention_map
        return output

In [30]:
# ==========================================
# 4. STEM ENHANCER (Your U-Net)
# ==========================================
class STEMEnhancementNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64, num_sfin=2, num_rhag=3, window_size=8):
        super().__init__()
        self.channels = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        
        self.input_conv = nn.Conv2d(in_channels, self.channels[0], 3, 1, 1)
        
        self.encoder1 = SFINBlock(self.channels[0], self.channels[0], num_sfin)
        self.down1 = nn.Conv2d(self.channels[0], self.channels[1], 3, stride=2, padding=1)
        self.encoder2 = SFINBlock(self.channels[1], self.channels[1], num_sfin)
        self.down2 = nn.Conv2d(self.channels[1], self.channels[2], 3, stride=2, padding=1)
        self.encoder3 = SFINBlock(self.channels[2], self.channels[2], num_sfin)
        self.down3 = nn.Conv2d(self.channels[2], self.channels[3], 3, stride=2, padding=1)
        self.encoder4 = SFINBlock(self.channels[3], self.channels[3], num_sfin)
        self.down4 = nn.Conv2d(self.channels[3], self.channels[3], 3, stride=2, padding=1)
        
        self.bottleneck = RHAG(dim=self.channels[3], num_heads=8, num_blocks=num_rhag, window_size=window_size)
        
        self.up1 = nn.ConvTranspose2d(self.channels[3], self.channels[3], 4, stride=2, padding=1)
        self.ag1 = AttentionGate(F_g=self.channels[3], F_l=self.channels[3], F_int=self.channels[2])
        self.decoder1 = SFINBlock(self.channels[3] * 2, self.channels[3], num_sfin)
        
        self.up2 = nn.ConvTranspose2d(self.channels[3], self.channels[2], 4, stride=2, padding=1)
        self.ag2 = AttentionGate(F_g=self.channels[2], F_l=self.channels[2], F_int=self.channels[1])
        self.decoder2 = SFINBlock(self.channels[2] * 2, self.channels[2], num_sfin)
        
        self.up3 = nn.ConvTranspose2d(self.channels[2], self.channels[1], 4, stride=2, padding=1)
        self.ag3 = AttentionGate(F_g=self.channels[1], F_l=self.channels[1], F_int=self.channels[0])
        self.decoder3 = SFINBlock(self.channels[1] * 2, self.channels[1], num_sfin)
        
        self.up4 = nn.ConvTranspose2d(self.channels[1], self.channels[0], 4, stride=2, padding=1)
        self.ag4 = AttentionGate(F_g=self.channels[0], F_l=self.channels[0], F_int=self.channels[0] // 2)
        self.decoder4 = SFINBlock(self.channels[0] * 2, self.channels[0], num_sfin)
        
        self.output_conv = nn.Conv2d(self.channels[0], out_channels, 3, 1, 1)
        
        # NOTE: Added projections to match your original GitHub architecture
        self.bottleneck_proj = nn.Conv2d(self.channels[3], out_channels, 1, 1, 0)
        self.decoder_proj = nn.Conv2d(self.channels[0], out_channels, 1, 1, 0)
        
    def forward(self, x, return_intermediate=False): # NOTE: Missing argument added here!
        x = self.input_conv(x)
        enc1 = self.encoder1(x)                    
        enc2 = self.encoder2(self.down1(enc1))                    
        enc3 = self.encoder3(self.down2(enc2))                    
        enc4 = self.encoder4(self.down3(enc3))                    
        
        bottleneck_feat = self.bottleneck(self.down4(enc4))       
        
        x = self.decoder1(torch.cat([self.up1(bottleneck_feat), self.ag1(self.up1(bottleneck_feat), enc4)], dim=1))
        x = self.decoder2(torch.cat([self.up2(x), self.ag2(self.up2(x), enc3)], dim=1))
        x = self.decoder3(torch.cat([self.up3(x), self.ag3(self.up3(x), enc2)], dim=1))
        decoder_feat = self.decoder4(torch.cat([self.up4(x), self.ag4(self.up4(x), enc1)], dim=1))            
        
        output = self.output_conv(decoder_feat)    
        
        # NOTE: Missing return logic added here!
        if return_intermediate:
            return {
                'bottleneck': self.bottleneck_proj(bottleneck_feat),
                'decoder': self.decoder_proj(decoder_feat),
                'final': output
            }
        return output

In [ ]:
# ==========================================
# 5. THE ULTIMATE DINO + SFIN WRAPPER
# ==========================================
class DINOSFIN_Architecture(nn.Module):
    def __init__(self, strategy="concat"):
        super().__init__()
        self.strategy = strategy
        print(f"⚙️ Compiling Complete SFIN Architecture: Strategy = {strategy.upper()}")
        
        # 1. The DINO Engine
        self.encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for param in self.encoder.parameters():
            param.requires_grad = False 
            
        # 2. Early Fusion Setup
        # DINO gives 384 channels. We will concatenate the 3 RGB channels from the image.
        dino_channels = 384
        rgb_channels = 3
        
        # We MUST use window_size=7 because 224 resolution downsampled 4 times is 14. (14 % 7 = 0)
        safe_window_size = 7 

        if strategy == "multi_head":
            in_c = dino_channels + rgb_channels
            self.decoder_head1 = STEMEnhancementNet(in_channels=in_c, window_size=safe_window_size)
            self.decoder_head2 = STEMEnhancementNet(in_channels=in_c, window_size=safe_window_size)
            
        elif strategy == "concat":
            # Add 1 more channel for the Layer Prompt Index
            in_c = dino_channels + rgb_channels + 1
            self.prompt_proj = nn.Linear(1, 1) 
            self.decoder = STEMEnhancementNet(in_channels=in_c, window_size=safe_window_size)

    def forward(self, x, target_layer=1):
        # 1. Extract DINO Features (16x16 grid)
        features = self.encoder.forward_features(x)["x_norm_patchtokens"]
        B, N, C = features.shape
        H = W = int(np.sqrt(N))
        dino_feats = features.permute(0, 2, 1).view(B, C, H, W)
        
        # 2. Upsample DINO features to 224x224 to match the RGB image
        dino_upsampled = F.interpolate(dino_feats, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        if self.strategy == "multi_head":
            # Fuse RGB + DINO
            fused_input = torch.cat([x, dino_upsampled], dim=1)
            raw_depth = self.decoder_head1(fused_input) if target_layer == 1 else self.decoder_head2(fused_input)
            
        elif self.strategy == "concat":
            # Fuse RGB + DINO + Prompt Map
            prompt = torch.tensor([[float(target_layer)]]).to(x.device)
            p_vec = self.prompt_proj(prompt).view(B, 1, 1, 1).expand(-1, -1, x.shape[2], x.shape[3])
            
            fused_input = torch.cat([x, dino_upsampled, p_vec], dim=1) 
            raw_depth = self.decoder(fused_input)
            
        # 3. Softplus enforces physical depth constraint (> 0)
        return F.softplus(raw_depth) 

# Test it
test_model = DINOSFIN_Architecture(strategy="concat").to(device)
print("✅ SFIN+DINO Network compiled perfectly without tensor shape errors!")

⚙️ Compiling Complete SFIN Architecture: Strategy = CONCAT


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main


✅ SFIN+DINO Network compiled perfectly without tensor shape errors!


In [12]:
import torch.optim as optim

def run_sfin_experiment(strategy_name, epochs=1, steps=500):
    print(f"\n🚀 Starting {strategy_name.upper()} Experiment with SFIN-UNet...")
    
    model = DINOSFIN_Architecture(strategy=strategy_name).to(device)
    
    # Optimize your SFIN U-Net (DINO is safely frozen)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    criterion = SILogLoss()
    
    model.train()
    for epoch in range(epochs):
        step = 0
        loop = tqdm(train_dataset, total=steps, desc=f"Epoch {epoch+1}")
        
        for sample in loop:
            img_pil = sample['image.png'].convert("RGB")
            img_t = transform_img(img_pil).unsqueeze(0).to(device)
            
            # Ground truth layers
            d1_target = transform_depth(sample['depth_1.png']).unsqueeze(0).to(device)
            d2_target = transform_depth(sample['depth_2.png']).unsqueeze(0).to(device)
            
            optimizer.zero_grad()
            
            # Predict
            p1 = model(img_t, target_layer=1)
            p2 = model(img_t, target_layer=2)
            
            # Loss Calculation
            loss1 = criterion(p1, d1_target)
            loss2 = criterion(p2, d2_target)
            total_loss = loss1 + loss2
            
            total_loss.backward()
            optimizer.step()
            
            loop.set_postfix(loss=total_loss.item())
            
            step += 1
            if step >= steps: 
                break
                
    print(f"✅ {strategy_name.upper()} Training Complete.")
    return model

# 1. Train both variations! (You can increase 'steps' for better accuracy later)
trained_sfin_concat = run_sfin_experiment("concat", steps=500)
trained_sfin_multihead = run_sfin_experiment("multi_head", steps=500)

# 2. SAVE YOUR SOTA WEIGHTS
torch.save(trained_sfin_concat.state_dict(), "sfin_concat_weights.pth")
print("✅ Weights safely stored.")


🚀 Starting CONCAT Experiment with SFIN-UNet...
⚙️ Compiling Complete SFIN Architecture: Strategy = CONCAT


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main
Epoch 1:   0%|          | 0/500 [00:00<?, ?it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/princeton-vl/LayeredDepth-Syn/resolve/78fd900929879332e60d7190d9bd423b8432669b/data/train-00000-of-00056.parquet
Retrying in 1s [Retry 1/5].
Epoch 1:   0%|          | 0/500 [01:09<?, ?it/s]


KeyboardInterrupt: 

In [13]:
import os
import torch

# 1. The Quadruplet Evaluator
def evaluate_image_tuples(predicted_depth_map, ground_truth_sample, original_size):
    correct, total = 0, 0
    try:
        annotation_key = 'tuples.json' if 'tuples.json' in ground_truth_sample else list(ground_truth_sample.keys())[-1]
        pairs_list = ground_truth_sample[annotation_key]['layer_all']['pairs']
    except KeyError:
        return 0, 0 
        
    orig_w, orig_h = original_size
    pred_h, pred_w = predicted_depth_map.shape[2:]
    scale_x, scale_y = pred_w / orig_w, pred_h / orig_h
    depth_array = predicted_depth_map[0, 0].cpu().numpy()

    for pair in pairs_list:
        if not pair.get('is_real', True): continue
        p1, p2 = pair['p1'], pair['p2']
        if p1[2] == p2[2]: continue 
            
        gt_rel = 1 if p1[2] < p2[2] else -1
        
        px1 = min(max(int(p1[0] * scale_x), 0), pred_w - 1)
        py1 = min(max(int(p1[1] * scale_y), 0), pred_h - 1)
        px2 = min(max(int(p2[0] * scale_x), 0), pred_w - 1)
        py2 = min(max(int(p2[1] * scale_y), 0), pred_h - 1)
        
        d1, d2 = depth_array[py1, px1], depth_array[py2, px2]
        pred_rel = 1 if d1 < d2 else -1
        
        if pred_rel == gt_rel: correct += 1
        total += 1
        
    return correct, total

# 2. Download Real-World Validation
print("⏳ Loading Real-World Validation Dataset...")
val_dataset = load_dataset("princeton-vl/LayeredDepth", split="validation", streaming=True)

# 3. The Evaluation Engine
def evaluate_sfin_model(model, limit=50):
    print(f"\n🚀 Running Evaluation on Real-World Quadruplets...")
    model.eval()
    total_correct = 0
    total_pairs = 0
    
    with torch.no_grad():
        for i, sample in enumerate(val_dataset):
            if i >= limit: break
            
            # Safe key extraction (handles both 'image' and 'image.png')
            img_key = 'image.png' if 'image.png' in sample else 'image'
            img_pil = sample[img_key].convert("RGB")
            
            img_t = transform_img(img_pil).unsqueeze(0).to(device)
            
            # Predict the Front Layer (Glass)
            pred_depth = model(img_t, target_layer=1)
            
            correct, total = evaluate_image_tuples(pred_depth, sample, img_pil.size)
            total_correct += correct
            total_pairs += total
            
            if (i+1) % 10 == 0:
                print(f"Evaluated {i+1} images...")

    final_acc = (total_correct / total_pairs) * 100 if total_pairs > 0 else 0
    print("\n" + "="*50)
    print(f"Total Valid Point Pairs Evaluated: {total_pairs}")
    print(f"Final Architecture Accuracy: {final_acc:.2f}%")
    print("="*50)
    return final_acc

# ==========================================
# 4. LOAD THE SAVED WEIGHTS AND RUN
# ==========================================
weights_path = "sfin_concat_weights.pth"

if os.path.exists(weights_path):
    print(f"⏳ Initializing SFIN Architecture to load '{weights_path}'...")
    # Initialize a fresh, untrained model
    eval_model = DINOSFIN_Architecture(strategy="concat").to(device)
    
    # Inject your saved weights into the model
    eval_model.load_state_dict(torch.load(weights_path, map_location=device))
    print("✅ Weights loaded perfectly!")
    
    # Run the evaluation!
    evaluate_sfin_model(eval_model, limit=50)
else:
    print(f"❌ Error: Could not find '{weights_path}' in your folder. Did the training cell finish?")

⏳ Loading Real-World Validation Dataset...


Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

⏳ Initializing SFIN Architecture to load 'sfin_concat_weights.pth'...
⚙️ Compiling Complete SFIN Architecture: Strategy = CONCAT


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main


✅ Weights loaded perfectly!

🚀 Running Evaluation on Real-World Quadruplets...
Evaluated 10 images...
Evaluated 20 images...
Evaluated 30 images...
Evaluated 40 images...
Evaluated 50 images...

Total Valid Point Pairs Evaluated: 7934
Final Architecture Accuracy: 57.51%


# Complete train pipeline

In [ ]:
# ==========================================
# 5. UPDATED DINO + SFIN WRAPPER (For Multi-Stage Loss)
# ==========================================
class DINOSFIN_Architecture_NEW(nn.Module):
    def __init__(self, strategy="concat"):
        super().__init__()
        self.strategy = strategy
        
        self.encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for param in self.encoder.parameters():
            param.requires_grad = False 
            
        safe_window_size = 7 
        in_c = 384 + 3 + 1 # DINO + RGB + Prompt
        
        self.prompt_proj = nn.Linear(1, 1) 
        self.decoder = STEMEnhancementNet(in_channels=in_c, window_size=safe_window_size)

    def forward(self, x, target_layer=1, return_intermediate=False):
        features = self.encoder.forward_features(x)["x_norm_patchtokens"]
        B, N, C = features.shape
        H = W = int(math.sqrt(N))
        
        dino_feats = features.permute(0, 2, 1).view(B, C, H, W)
        dino_upsampled = F.interpolate(dino_feats, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        # FIX: Expand prompt to match batch size B
        prompt = torch.tensor([[float(target_layer)]], device=x.device).expand(B, 1)
        
        p_vec = self.prompt_proj(prompt).view(B, 1, 1, 1).expand(-1, -1, x.shape[2], x.shape[3])
        fused_input = torch.cat([x, dino_upsampled, p_vec], dim=1) 
        
        if return_intermediate:
            out_dict = self.decoder(fused_input, return_intermediate=True)
            return {
                'bottleneck': F.softplus(out_dict['bottleneck']),
                'decoder': F.softplus(out_dict['decoder']),
                'final': F.softplus(out_dict['final'])
            }
        return F.softplus(self.decoder(fused_input))

In [24]:
import os
import torch
from datasets import load_dataset

def evaluate_image_tuples(predicted_depth_map, ground_truth_sample, original_size):
    correct, total = 0, 0
    try:
        annotation_key = 'tuples.json' if 'tuples.json' in ground_truth_sample else list(ground_truth_sample.keys())[-1]
        pairs_list = ground_truth_sample[annotation_key]['layer_all']['pairs']
    except KeyError:
        return 0, 0 
        
    orig_w, orig_h = original_size
    pred_h, pred_w = predicted_depth_map.shape[2:]
    scale_x, scale_y = pred_w / orig_w, pred_h / orig_h
    depth_array = predicted_depth_map[0, 0].cpu().numpy()

    for pair in pairs_list:
        if not pair.get('is_real', True): continue
        p1, p2 = pair['p1'], pair['p2']
        if p1[2] == p2[2]: continue 
            
        gt_rel = 1 if p1[2] < p2[2] else -1
        
        px1 = min(max(int(p1[0] * scale_x), 0), pred_w - 1)
        py1 = min(max(int(p1[1] * scale_y), 0), pred_h - 1)
        px2 = min(max(int(p2[0] * scale_x), 0), pred_w - 1)
        py2 = min(max(int(p2[1] * scale_y), 0), pred_h - 1)
        
        d1, d2 = depth_array[py1, px1], depth_array[py2, px2]
        pred_rel = 1 if d1 < d2 else -1
        
        if pred_rel == gt_rel: correct += 1
        total += 1
        
    return correct, total

def evaluate_sfin_model(model, limit=None):
    print(f"\n🚀 Running Complete Real-World Evaluation...")
    model.eval()
    total_correct = 0
    total_pairs = 0
    
    val_dataset = load_dataset("princeton-vl/LayeredDepth", split="validation", streaming=True)
    
    with torch.no_grad():
        for i, sample in enumerate(val_dataset):
            if limit and i >= limit: break
            
            img_key = 'image.png' if 'image.png' in sample else 'image'
            img_pil = sample[img_key].convert("RGB")
            img_t = transform_img(img_pil).unsqueeze(0).to(device)
            
            pred_depth = model(img_t, target_layer=1, return_intermediate=False)
            
            correct, total = evaluate_image_tuples(pred_depth, sample, img_pil.size)
            total_correct += correct
            total_pairs += total
            
            # Print an update every 100 images since we are doing the full dataset
            if (i+1) % 100 == 0:
                print(f"Evaluated {i+1} images...")

    final_acc = (total_correct / total_pairs) * 100 if total_pairs > 0 else 0
    print("\n" + "="*50)
    print(f"Total Valid Point Pairs Evaluated: {total_pairs}")
    print(f"Final Architecture Accuracy: {final_acc:.2f}%")
    print("="*50)
    return final_acc

In [33]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from tqdm import tqdm

def collate_fn(batch):
    images = [transform_img(b['image.png' if 'image.png' in b else 'image'].convert("RGB")) for b in batch]
    depth_1s = [transform_depth(b['depth_1.png']) for b in batch]
    depth_2s = [transform_depth(b['depth_2.png']) for b in batch]
    
    return {
        'image': torch.stack(images),
        'depth_1': torch.stack(depth_1s),
        'depth_2': torch.stack(depth_2s)
    }

def train_production_model(epochs=20, batch_size=4, accum_steps=4):
    print(f"\n🔥 Initiating RTX 4060 Production Training")
    print(f"Batch Size: {batch_size} | Gradient Accumulation Steps: {accum_steps} | Effective Batch: {batch_size * accum_steps}")
    
    os.makedirs("checkpoints", exist_ok=True)
    
    model = DINOSFIN_Architecture_NEW(strategy="concat").to(device)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    criterion = SILogLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    scaler = GradScaler()
    
    print("⏳ Loading Streaming Dataset...")
    train_dataset = load_dataset("princeton-vl/LayeredDepth-Syn", split="train", streaming=True)
    
    # num_workers=2 for your 8-core CPU. (Change to 0 if WSL crashes).
    train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=2)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        step_count = 0
        optimizer.zero_grad()
        
        # We wrap train_loader in tqdm to track the whole epoch
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for i, batch in enumerate(loop):
            img = batch['image'].to(device)
            d1_tgt = batch['depth_1'].to(device)
            d2_tgt = batch['depth_2'].to(device)
            
            with autocast('cuda'):
                out1 = model(img, target_layer=1, return_intermediate=True)
                out2 = model(img, target_layer=2, return_intermediate=True)
                
                loss1 = criterion(out1['final'], d1_tgt) + (0.5 * criterion(out1['decoder'], d1_tgt)) + (0.25 * criterion(out1['bottleneck'], d1_tgt))
                loss2 = criterion(out2['final'], d2_tgt) + (0.5 * criterion(out2['decoder'], d2_tgt)) + (0.25 * criterion(out2['bottleneck'], d2_tgt))
                
                combined_loss = (loss1 + loss2) / accum_steps
            
            scaler.scale(combined_loss).backward()
            
            if (i + 1) % accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            epoch_loss += combined_loss.item() * accum_steps
            step_count += 1
            loop.set_postfix(loss=(epoch_loss / step_count))
        
        avg_epoch_loss = epoch_loss / step_count
        print(f"\n📊 Epoch {epoch+1} Complete | Average Loss: {avg_epoch_loss:.4f}")
        scheduler.step(avg_epoch_loss)
        
        if avg_epoch_loss < best_loss:
            best_loss = avg_epoch_loss
            save_path = f"checkpoints/sfin_concat_best_NEW.pth"
            torch.save(model.state_dict(), save_path)
            print(f"⭐ Best Model Saved to {save_path}!")
            
        # Run Real-World Evaluation every 5 epochs using the FULL dataset
        if (epoch + 1) % 5 == 0:
            evaluate_sfin_model(model, limit=None)

    print("✅ Production Training Finished.")
    return model

# START TRAINING
trained_model = train_production_model(epochs=20, batch_size=2, accum_steps=8)


🔥 Initiating RTX 4060 Production Training
Batch Size: 2 | Gradient Accumulation Steps: 8 | Effective Batch: 16


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main
/tmp/ipykernel_3986/3083521054.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


⏳ Loading Streaming Dataset...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Epoch 1/20: 8it [04:04, 30.52s/it, loss=17.2]


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 8. FINAL OFFICIAL EVALUATION
# ==========================================
weights_path = "checkpoints/sfin_concat_best_NEW.pth"

if os.path.exists(weights_path):
    print(f"⏳ Loading the BEST trained weights from '{weights_path}'...")
    
    # 1. Initialize a fresh model
    final_eval_model = DINOSFIN_Architecture_NEW(strategy="concat").to(device)
    
    # 2. Load your heavily trained weights
    final_eval_model.load_state_dict(torch.load(weights_path, map_location=device))
    print("✅ Weights loaded perfectly!")
    
    # 3. Run the complete evaluation on the entire real-world dataset
    evaluate_sfin_model(final_eval_model, limit=None)
else:
    print(f"❌ Error: Could not find '{weights_path}'. Ensure training finished successfully.")